In [20]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Machine learning & NLP libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)

# Deep learning libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

# Add parent directory to path for config imports
sys.path.insert(0, os.path.abspath('..'))
from config import unsloth_gemma4_model, embedding_model

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

PyTorch version: 2.12.0+cpu
GPU available: False
Device: cpu


# CLINC_OOS Fine-tuning with Query Intent Classification

This notebook implements:
1. CLINC_OOS dataset loading and exploration
2. Query categorization: **text_only** (label 0) vs **dynamic_retrieval_intent** (label 1)
3. Baseline model: Logistic Regression + TF-IDF
4. Fine-tuned model: BERT/DistilBERT transformer
5. Comprehensive evaluation: accuracy, precision, recall, F1-score, BLEU, perplexity

**Query Categorization:**
- **text_only (label 0)**: Informational/reference domains (meaning_of_life, what_is_your_name, who_made_you, definition queries) + entire out_of_scope split
- **dynamic_retrieval_intent (label 1)**: Queries requiring code execution or online data fetching

## Section 1: Load and Explore CLINC_OOS Dataset

Load the CLINC_OOS (Crowdsourced Lightweight Intent Classification Benchmark for Out-of-Scope Detection) dataset from HuggingFace Datasets.

In [21]:
# Load CLINC_OOS dataset from HuggingFace
print("Loading CLINC_OOS dataset...")
dataset = load_dataset("clinc/clinc_oos", "plus")

# Display dataset structure
print(f"\nDataset splits: {list(dataset.keys())}")
print(f"\nTrain set size: {len(dataset['train'])}")
print(f"Valid set size:  {len(dataset['validation'])}")
print(f"Test set size:   {len(dataset['test'])}")

# Examine first few samples
print("\n=== Sample from training set ===")
for i in range(3):
    sample = dataset['train'][i]
    print(f"\nSample {i+1}:")
    print(f"  Text:   {sample['text']}")
    print(f"  Intent: {sample['intent']}")

# Get intent label names
intent_names = dataset['train'].features['intent'].names

# Build DataFrame with readable labels
train_data = pd.DataFrame(dataset['train'])
train_data['intent_name'] = train_data['intent'].map(lambda x: intent_names[x])

print(f"\n=== Dataset Statistics ===")
print(f"Unique intents: {train_data['intent'].nunique()}")

print(f"\nIntent distribution (top 15):")
print(train_data['intent_name'].value_counts().head(15))

Loading CLINC_OOS dataset...

Dataset splits: ['train', 'validation', 'test']

Train set size: 15250
Valid set size:  3100
Test set size:   5500

=== Sample from training set ===

Sample 1:
  Text:   what expression would i use to say i love you if i were an italian
  Intent: 61

Sample 2:
  Text:   can you tell me how to say 'i do not speak much spanish', in spanish
  Intent: 61

Sample 3:
  Text:   what is the equivalent of, 'life is good' in french
  Intent: 61

=== Dataset Statistics ===
Unique intents: 151

Intent distribution (top 15):
intent_name
oos                          250
translate                    100
transfer                     100
timer                        100
definition                   100
meaning_of_life              100
insurance_change             100
find_phone                   100
travel_alert                 100
pto_request                  100
improve_credit_score         100
fun_fact                     100
change_language              100
payday     

## Section 2: Preprocess and Create Query Intent Labels

Create binary labels:
- **Label 0 (text_only)**: Queries from informational/reference domains (meaning_of_life, what_is_your_name, who_made_you, definition queries) + entire out_of_scope split
- **Label 1 (dynamic_retrieval_intent)**: Queries requiring code execution or online data fetching

In [ ]:
# Get intent label names from dataset features
intent_names = dataset['train'].features['intent'].names

# Define text_only intents (informational/reference domains)
TEXT_ONLY_INTENTS = {
    'meaning_of_life',
    'what_is_your_name',
    'who_made_you',
    'definition',
    'oos',
}

# Define dynamic_retrieval_intent categories
DYNAMIC_RETRIEVAL_INTENT_KEYWORDS = [
    'weather', 'traffic', 'schedule', 'book', 'order', 'buy', 'pay',
    'set', 'turn', 'change', 'update', 'create', 'delete', 'send',
    'call', 'text', 'email', 'play', 'stream', 'search', 'find',
    'cancel', 'refund', 'delivery', 'track', 'rate', 'review',
    'timer', 'alarm', 'reminder', 'calendar', 'appointment'
]

def categorize_intent(intent_id: int) -> int:
    """
    Categorize intent into:
    0 = text_only
    1 = dynamic_retrieval_intent
    """
    intent_str = intent_names[intent_id].lower()  # decode int -> string

    if intent_str in TEXT_ONLY_INTENTS:
        return 0

    for keyword in DYNAMIC_RETRIEVAL_INTENT_KEYWORDS:
        if keyword in intent_str:
            return 1

    return 1  # default to dynamic

# Apply categorization to all splits
def add_binary_labels(dataset):
    processed_data = []
    for split_name in dataset.keys():
        data_list = []
        for sample in dataset[split_name]:
            sample_copy = dict(sample)
            intent_id = sample['intent']
            sample_copy['intent_name'] = intent_names[intent_id]   # add readable name
            sample_copy['binary_label'] = categorize_intent(intent_id)
            data_list.append(sample_copy)
        processed_data.append((split_name, data_list))
    return processed_data

processed_splits = add_binary_labels(dataset)

# Create processed DataFrames
processed_dataset = {}
for split_name, data_list in processed_splits:
    processed_dataset[split_name] = pd.DataFrame(data_list)

print("Binary labels created!")
print("\n=== Label Distribution ===")
for split_name, df in processed_dataset.items():
    n_text    = (df['binary_label'] == 0).sum()
    n_dynamic = (df['binary_label'] == 1).sum()
    total     = len(df)
    print(f"\n{split_name.upper()} set:")
    print(f"  Label 0 (text_only):              {n_text:4d} ({n_text/total*100:.1f}%)")
    print(f"  Label 1 (dynamic_retrieval_intent): {n_dynamic:4d} ({n_dynamic/total*100:.1f}%)")

# Display sample mappings using intent_name column
print("\n=== Sample Intent Mappings ===")
sample_rows = processed_dataset['train'].drop_duplicates('intent').head(10)
for _, row in sample_rows.iterrows():
    label_name = 'text_only' if row['binary_label'] == 0 else 'dynamic_retrieval_intent'
    print(f"Intent: {row['intent_name']:35} | Label: {row['binary_label']} ({label_name})")

AttributeError: 'int' object has no attribute 'lower'

## Section 3: Prepare Data for Model Training

Split data into training, validation, and test sets. Prepare data for both baseline and fine-tuned models.

In [ ]:
# Prepare datasets
train_df = processed_dataset['train']
valid_df = processed_dataset['validation']
test_df = processed_dataset['test']

# For demonstration, we can use a subset if needed (remove for full dataset)
# train_df = train_df.sample(n=5000, random_state=42)
# valid_df = valid_df.sample(n=500, random_state=42)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(valid_df)}")
print(f"Test set size: {len(test_df)}")

# Extract texts and labels
train_texts = train_df['text'].values
train_labels = train_df['binary_label'].values

valid_texts = valid_df['text'].values
valid_labels = valid_df['binary_label'].values

test_texts = test_df['text'].values
test_labels = test_df['binary_label'].values

print("\nData prepared successfully!")
print(f"Sample training texts:")
for i in range(3):
    label_name = 'text_only' if train_labels[i] == 0 else 'dynamic_retrieval_intent'
    print(f"  [{label_name}] {train_texts[i][:80]}")

# Check for class imbalance
print("\n=== Class Balance Analysis ===")
unique, counts = np.unique(train_labels, return_counts=True)
for label, count in zip(unique, counts):
    label_name = 'text_only' if label == 0 else 'dynamic_retrieval_intent'
    print(f"Label {label} ({label_name}): {count} ({count/len(train_labels)*100:.1f}%)")

# Calculate class weights for handling imbalance
class_weights = len(train_labels) / (len(np.unique(train_labels)) * np.bincount(train_labels))
print(f"\nClass weights: {class_weights}")

## Section 4: Implement Baseline Models (Logistic Regression + TF-IDF)

Train a baseline model using TF-IDF vectorization and Logistic Regression.

In [ ]:
# TF-IDF Vectorization
print("Training baseline model (TF-IDF + Logistic Regression)...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    lowercase=True,
    stop_words='english'
)

# Fit on training data
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_valid_tfidf = tfidf_vectorizer.transform(valid_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

print(f"TF-IDF features shape: {X_train_tfidf.shape}")

# Train Logistic Regression baseline
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
baseline_model.fit(X_train_tfidf, train_labels)

# Generate predictions
baseline_train_preds = baseline_model.predict(X_train_tfidf)
baseline_valid_preds = baseline_model.predict(X_valid_tfidf)
baseline_test_preds = baseline_model.predict(X_test_tfidf)

# Store baseline results
baseline_results = {
    'train': baseline_train_preds,
    'valid': baseline_valid_preds,
    'test': baseline_test_preds,
    'model': baseline_model,
    'vectorizer': tfidf_vectorizer
}

print("Baseline model training completed!")
print(f"\nBaseline predictions on test set (first 10):")
print(baseline_test_preds[:10])
print(f"Actual labels (first 10):")
print(test_labels[:10])

## Section 5: Fine-tune Transformer Model

Fine-tune a pre-trained transformer model (DistilBERT or BERT) for binary intent classification.

In [ ]:
# Create custom Dataset class
class BinaryIntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Load tokenizer and model
print("Loading transformer model...")
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    problem_type='single_label_classification'
)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model moved to {device}")

# Create datasets
print("Creating datasets...")
train_dataset = BinaryIntentDataset(train_texts, train_labels, tokenizer)
valid_dataset = BinaryIntentDataset(valid_texts, valid_labels, tokenizer)
test_dataset = BinaryIntentDataset(test_texts, test_labels, tokenizer)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./finetuned_model',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    push_to_hub=False,
)

# Define evaluation metrics function
def compute_metrics(p):
    pred_labels = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, pred_labels)
    precision = precision_score(p.label_ids, pred_labels, zero_division=0)
    recall = recall_score(p.label_ids, pred_labels, zero_division=0)
    f1 = f1_score(p.label_ids, pred_labels, zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

# Fine-tune the model
print("\nStarting fine-tuning...")
train_result = trainer.train()

print(f"\nFine-tuning completed!")
print(f"Training loss: {train_result.training_loss:.4f}")

# Save the fine-tuned model
model_save_path = './finetuned_intent_model'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")

## Section 6: Evaluate Models with Multiple Metrics

Calculate comprehensive evaluation metrics including accuracy, precision, recall, F1-score, BLEU score, and perplexity.

In [ ]:
# Get predictions from fine-tuned model on test set
print("Generating predictions from fine-tuned model...")
model.eval()

def get_model_predictions(texts, labels, tokenizer, model, device, batch_size=32):
    """Get predictions from fine-tuned model"""
    all_preds = []
    all_logits = []
    
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            encoding = tokenizer(
                list(batch_texts),
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            # Move to device
            input_ids = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            
            # Get predictions
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_logits.extend(logits.cpu().numpy())
    
    return np.array(all_preds), np.array(all_logits)

finetuned_train_preds, train_logits = get_model_predictions(train_texts, train_labels, tokenizer, model, device)
finetuned_valid_preds, valid_logits = get_model_predictions(valid_texts, valid_labels, tokenizer, model, device)
finetuned_test_preds, test_logits = get_model_predictions(test_texts, test_labels, tokenizer, model, device)

# Calculate BLEU score (for classification, we use label sequences)
def calculate_bleu(y_true, y_pred):
    """Calculate BLEU score for classification"""
    # Convert labels to sequences
    reference = [[str(label)] for label in y_true]
    hypothesis = [str(label) for label in y_pred]
    
    bleu_scores = []
    for ref, hyp in zip(reference, hypothesis):
        # BLEU-1 for single tokens
        score = sentence_bleu(ref, hyp.split(), smoothing_function=SmoothingFunction().method1)
        bleu_scores.append(score)
    
    return np.mean(bleu_scores)

# Calculate Perplexity
def calculate_perplexity(y_true, logits):
    """Calculate perplexity based on model confidence"""
    # Softmax for probabilities
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    
    # Get probability of correct class
    correct_probs = probs[np.arange(len(y_true)), y_true]
    
    # Perplexity = exp(-mean(log(prob)))
    log_probs = np.log(np.clip(correct_probs, 1e-7, 1.0))
    perplexity = np.exp(-np.mean(log_probs))
    
    return perplexity

# Calculate metrics for both models
def evaluate_model(y_true, y_pred, logits, model_name):
    """Evaluate model with all metrics"""
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'bleu': calculate_bleu(y_true, y_pred),
        'perplexity': calculate_perplexity(y_true, logits) if logits is not None else 0.0,
    }
    return metrics

# Evaluate Baseline
print("\n=== BASELINE MODEL EVALUATION ===")
baseline_test_logits = np.zeros((len(test_labels), 2))  # Dummy logits for baseline
baseline_metrics = evaluate_model(test_labels, baseline_test_preds, baseline_test_logits, 'Logistic Regression + TF-IDF')

print(f"Accuracy:   {baseline_metrics['accuracy']:.4f}")
print(f"Precision:  {baseline_metrics['precision']:.4f}")
print(f"Recall:     {baseline_metrics['recall']:.4f}")
print(f"F1-Score:   {baseline_metrics['f1']:.4f}")
print(f"BLEU Score: {baseline_metrics['bleu']:.4f}")
print(f"Perplexity: {baseline_metrics['perplexity']:.4f}")

# Evaluate Fine-tuned Model
print("\n=== FINE-TUNED MODEL EVALUATION ===")
finetuned_metrics = evaluate_model(test_labels, finetuned_test_preds, test_logits, 'DistilBERT Fine-tuned')

print(f"Accuracy:   {finetuned_metrics['accuracy']:.4f}")
print(f"Precision:  {finetuned_metrics['precision']:.4f}")
print(f"Recall:     {finetuned_metrics['recall']:.4f}")
print(f"F1-Score:   {finetuned_metrics['f1']:.4f}")
print(f"BLEU Score: {finetuned_metrics['bleu']:.4f}")
print(f"Perplexity: {finetuned_metrics['perplexity']:.4f}")

# Confusion matrices
print("\n=== CONFUSION MATRICES ===")
print("\nBaseline Model:")
baseline_cm = confusion_matrix(test_labels, baseline_test_preds)
print(baseline_cm)

print("\nFine-tuned Model:")
finetuned_cm = confusion_matrix(test_labels, finetuned_test_preds)
print(finetuned_cm)

# Classification reports
print("\n=== CLASSIFICATION REPORTS ===")
print("\nBaseline Model:")
print(classification_report(test_labels, baseline_test_preds, target_names=['text_only', 'dynamic_retrieval']))

print("\nFine-tuned Model:")
print(classification_report(test_labels, finetuned_test_preds, target_names=['text_only', 'dynamic_retrieval']))

## Section 7: Compare Baseline vs Fine-tuned Model Performance

Create comprehensive comparison tables and visualizations.

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame([baseline_metrics, finetuned_metrics])
print("=== SIDE-BY-SIDE MODEL COMPARISON ===")
print(comparison_df.to_string(index=False))

# Create improvement metrics
improvements = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'bleu']:
    baseline_val = baseline_metrics[metric]
    finetuned_val = finetuned_metrics[metric]
    improvement = ((finetuned_val - baseline_val) / abs(baseline_val) * 100) if baseline_val != 0 else 0
    improvements[metric] = improvement

print("\n=== PERFORMANCE IMPROVEMENTS (%) ===")
for metric, improvement in improvements.items():
    direction = "↑" if improvement > 0 else "↓"
    print(f"{metric.upper():12}: {improvement:+.2f}% {direction}")

# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Baseline vs Fine-tuned Model Performance Comparison', fontsize=16, fontweight='bold')

# 1. Accuracy comparison
ax = axes[0, 0]
models = [baseline_metrics['model'], finetuned_metrics['model']]
accuracies = [baseline_metrics['accuracy'], finetuned_metrics['accuracy']]
bars = ax.bar(models, accuracies, color=['#FF6B6B', '#4ECDC4'])
ax.set_ylabel('Accuracy')
ax.set_ylim([0, 1])
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')
ax.set_title('Accuracy')

# 2. Precision comparison
ax = axes[0, 1]
precisions = [baseline_metrics['precision'], finetuned_metrics['precision']]
bars = ax.bar(models, precisions, color=['#FF6B6B', '#4ECDC4'])
ax.set_ylabel('Precision')
ax.set_ylim([0, 1])
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')
ax.set_title('Precision')

# 3. Recall comparison
ax = axes[0, 2]
recalls = [baseline_metrics['recall'], finetuned_metrics['recall']]
bars = ax.bar(models, recalls, color=['#FF6B6B', '#4ECDC4'])
ax.set_ylabel('Recall')
ax.set_ylim([0, 1])
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')
ax.set_title('Recall')

# 4. F1-Score comparison
ax = axes[1, 0]
f1_scores = [baseline_metrics['f1'], finetuned_metrics['f1']]
bars = ax.bar(models, f1_scores, color=['#FF6B6B', '#4ECDC4'])
ax.set_ylabel('F1-Score')
ax.set_ylim([0, 1])
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')
ax.set_title('F1-Score')

# 5. BLEU Score comparison
ax = axes[1, 1]
bleu_scores = [baseline_metrics['bleu'], finetuned_metrics['bleu']]
bars = ax.bar(models, bleu_scores, color=['#FF6B6B', '#4ECDC4'])
ax.set_ylabel('BLEU Score')
ax.set_ylim([0, 1])
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')
ax.set_title('BLEU Score')

# 6. Perplexity comparison
ax = axes[1, 2]
perplexities = [baseline_metrics['perplexity'], finetuned_metrics['perplexity']]
bars = ax.bar(models, perplexities, color=['#FF6B6B', '#4ECDC4'])
ax.set_ylabel('Perplexity')
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')
ax.set_title('Perplexity (lower is better)')

plt.tight_layout()
plt.savefig('./model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nComparison plot saved as 'model_comparison.png'")

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

# Baseline confusion matrix
ax = axes[0]
sns.heatmap(baseline_cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
            xticklabels=['text_only', 'dynamic_retrieval'],
            yticklabels=['text_only', 'dynamic_retrieval'])
ax.set_title('Baseline Model (TF-IDF + Logistic Regression)')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

# Fine-tuned confusion matrix
ax = axes[1]
sns.heatmap(finetuned_cm, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['text_only', 'dynamic_retrieval'],
            yticklabels=['text_only', 'dynamic_retrieval'])
ax.set_title('Fine-tuned Model (DistilBERT)')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('./confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrices saved as 'confusion_matrices.png'")

## Summary and Insights

### Key Findings

1. **Model Performance**: Compare the effectiveness of the baseline and fine-tuned models
2. **Query Categorization**: Analyze how well models distinguish between text_only and dynamic_retrieval_intent queries
3. **Trade-offs**: Evaluate precision vs recall and other metrics
4. **Recommendations**: Based on evaluation results, provide recommendations for deployment

In [ ]:
# Generate insights and analysis
print("=" * 80)
print("ANALYSIS AND INSIGHTS")
print("=" * 80)

print("\n1. QUERY CATEGORIZATION SUMMARY:")
print(f"   - Text_only (label 0) queries: {(test_labels == 0).sum()} samples ({(test_labels == 0).sum()/len(test_labels)*100:.1f}%)")
print(f"   - Dynamic_retrieval_intent (label 1) queries: {(test_labels == 1).sum()} samples ({(test_labels == 1).sum()/len(test_labels)*100:.1f}%)")

print("\n2. MODEL COMPARISON:")
print(f"   Baseline Model: Logistic Regression + TF-IDF")
print(f"   Fine-tuned Model: DistilBERT")
print(f"   - Test Set Size: {len(test_labels)} samples")

print("\n3. PERFORMANCE METRICS:")
print(f"   {'Metric':<20} {'Baseline':<15} {'Fine-tuned':<15} {'Improvement':<15}")
print(f"   {'-'*65}")
print(f"   {'Accuracy':<20} {baseline_metrics['accuracy']:<15.4f} {finetuned_metrics['accuracy']:<15.4f} {improvements['accuracy']:+.2f}%")
print(f"   {'Precision':<20} {baseline_metrics['precision']:<15.4f} {finetuned_metrics['precision']:<15.4f} {improvements['precision']:+.2f}%")
print(f"   {'Recall':<20} {baseline_metrics['recall']:<15.4f} {finetuned_metrics['recall']:<15.4f} {improvements['recall']:+.2f}%")
print(f"   {'F1-Score':<20} {baseline_metrics['f1']:<15.4f} {finetuned_metrics['f1']:<15.4f} {improvements['f1']:+.2f}%")
print(f"   {'BLEU':<20} {baseline_metrics['bleu']:<15.4f} {finetuned_metrics['bleu']:<15.4f} {improvements['bleu']:+.2f}%")

print("\n4. CONFUSION MATRIX ANALYSIS:")
print(f"   Baseline Model:")
print(f"   - True Positives (TP): {baseline_cm[1, 1]}")
print(f"   - True Negatives (TN): {baseline_cm[0, 0]}")
print(f"   - False Positives (FP): {baseline_cm[0, 1]}")
print(f"   - False Negatives (FN): {baseline_cm[1, 0]}")

print(f"\n   Fine-tuned Model:")
print(f"   - True Positives (TP): {finetuned_cm[1, 1]}")
print(f"   - True Negatives (TN): {finetuned_cm[0, 0]}")
print(f"   - False Positives (FP): {finetuned_cm[0, 1]}")
print(f"   - False Negatives (FN): {finetuned_cm[1, 0]}")

print("\n5. RECOMMENDATIONS:")
if finetuned_metrics['f1'] > baseline_metrics['f1']:
    print(f"   ✓ Fine-tuned model shows better F1-score ({finetuned_metrics['f1']:.4f} vs {baseline_metrics['f1']:.4f})")
    print(f"   → Deploy the fine-tuned DistilBERT model for production use")
else:
    print(f"   ✓ Baseline model shows competitive performance")
    print(f"   → Consider using baseline for faster inference or fine-tuned for better accuracy")

if finetuned_metrics['accuracy'] > 0.85:
    print(f"   ✓ Model achieves high accuracy ({finetuned_metrics['accuracy']:.2%})")
    print(f"   → Model is suitable for production deployment")
else:
    print(f"   ⚠ Model accuracy could be improved ({finetuned_metrics['accuracy']:.2%})")
    print(f"   → Consider data augmentation, hyperparameter tuning, or using a larger model")

print("\n" + "=" * 80)

## Section 8: Export Fine-tuned Model Using Joblib

Save the fine-tuned transformer model and related artifacts using joblib for easy serialization and loading.

In [ ]:
import joblib
import json
from datetime import datetime

# Create export directory
export_dir = './exported_models'
os.makedirs(export_dir, exist_ok=True)

# Generate timestamp for versioning
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_version = f'intent_classifier_{timestamp}'
model_path = os.path.join(export_dir, model_version)
os.makedirs(model_path, exist_ok=True)

print(f"Exporting models to: {model_path}\n")

# ============= BASELINE MODEL EXPORT =============
print("=" * 70)
print("EXPORTING BASELINE MODEL (Logistic Regression + TF-IDF)")
print("=" * 70)

baseline_model_file = os.path.join(model_path, 'baseline_model.joblib')
baseline_vectorizer_file = os.path.join(model_path, 'baseline_vectorizer.joblib')

# Export baseline model
joblib.dump(baseline_model, baseline_model_file)
print(f"✓ Baseline model saved: {baseline_model_file}")

# Export TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, baseline_vectorizer_file)
print(f"✓ TF-IDF vectorizer saved: {baseline_vectorizer_file}")

# ============= FINE-TUNED MODEL EXPORT =============
print("\n" + "=" * 70)
print("EXPORTING FINE-TUNED MODEL (DistilBERT)")
print("=" * 70)

finetuned_model_dir = os.path.join(model_path, 'finetuned_transformer')
os.makedirs(finetuned_model_dir, exist_ok=True)

# Save model and tokenizer using Hugging Face methods
model.save_pretrained(os.path.join(finetuned_model_dir, 'model'))
tokenizer.save_pretrained(os.path.join(finetuned_model_dir, 'tokenizer'))
print(f"✓ Fine-tuned model saved: {os.path.join(finetuned_model_dir, 'model')}")
print(f"✓ Tokenizer saved: {os.path.join(finetuned_model_dir, 'tokenizer')}")

# ============= METADATA AND CONFIGURATION EXPORT =============
print("\n" + "=" * 70)
print("EXPORTING METADATA AND CONFIGURATION")
print("=" * 70)

# Create comprehensive metadata
metadata = {
    'model_version': model_version,
    'export_timestamp': timestamp,
    'dataset': 'CLINC_OOS',
    'binary_classification': {
        'label_0': 'text_only (informational/reference domains)',
        'label_1': 'dynamic_retrieval_intent (requires code execution/data fetching)',
        'text_only_intents': list(TEXT_ONLY_INTENTS),
        'dynamic_retrieval_keywords': DYNAMIC_RETRIEVAL_INTENT_KEYWORDS,
    },
    'data_split': {
        'train_size': len(train_labels),
        'valid_size': len(valid_labels),
        'test_size': len(test_labels),
        'label_distribution': {
            'text_only': int((test_labels == 0).sum()),
            'dynamic_retrieval_intent': int((test_labels == 1).sum()),
        }
    },
    'baseline_model': {
        'name': 'Logistic Regression + TF-IDF',
        'vectorizer_config': {
            'max_features': 5000,
            'ngram_range': [1, 2],
            'min_df': 2,
            'max_df': 0.95,
        },
        'model_file': 'baseline_model.joblib',
        'vectorizer_file': 'baseline_vectorizer.joblib',
    },
    'finetuned_model': {
        'name': 'DistilBERT',
        'pretrained_model': 'distilbert-base-uncased',
        'max_sequence_length': 128,
        'training_epochs': 3,
        'batch_size': 32,
        'learning_rate': 'default (5e-5)',
        'model_dir': 'finetuned_transformer/model',
        'tokenizer_dir': 'finetuned_transformer/tokenizer',
    },
    'performance_metrics': {
        'baseline': {
            'accuracy': float(baseline_metrics['accuracy']),
            'precision': float(baseline_metrics['precision']),
            'recall': float(baseline_metrics['recall']),
            'f1': float(baseline_metrics['f1']),
            'bleu': float(baseline_metrics['bleu']),
            'perplexity': float(baseline_metrics['perplexity']),
        },
        'finetuned': {
            'accuracy': float(finetuned_metrics['accuracy']),
            'precision': float(finetuned_metrics['precision']),
            'recall': float(finetuned_metrics['recall']),
            'f1': float(finetuned_metrics['f1']),
            'bleu': float(finetuned_metrics['bleu']),
            'perplexity': float(finetuned_metrics['perplexity']),
        }
    }
}

# Save metadata as JSON
metadata_file = os.path.join(model_path, 'metadata.json')
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Metadata saved: {metadata_file}")

# ============= HELPER FUNCTIONS EXPORT =============
print("\n" + "=" * 70)
print("EXPORTING HELPER FUNCTIONS AND UTILITIES")
print("=" * 70)

# Create utility module
utilities_code = '''"""
Utility functions for loading and using exported models.
"""
import joblib
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

def load_baseline_model(model_dir):
    """Load baseline model and vectorizer"""
    model = joblib.load(os.path.join(model_dir, 'baseline_model.joblib'))
    vectorizer = joblib.load(os.path.join(model_dir, 'baseline_vectorizer.joblib'))
    return model, vectorizer

def load_finetuned_model(model_dir, device='cpu'):
    """Load fine-tuned transformer model and tokenizer"""
    model_path = os.path.join(model_dir, 'finetuned_transformer', 'model')
    tokenizer_path = os.path.join(model_dir, 'finetuned_transformer', 'tokenizer')
    
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    
    model.to(device)
    model.eval()
    
    return model, tokenizer

def load_metadata(model_dir):
    """Load model metadata"""
    with open(os.path.join(model_dir, 'metadata.json'), 'r') as f:
        return json.load(f)

def predict_baseline(texts, model, vectorizer):
    """Get predictions from baseline model"""
    X = vectorizer.transform(texts)
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)
    return predictions, probabilities

def predict_finetuned(texts, model, tokenizer, device='cpu'):
    """Get predictions from fine-tuned model"""
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for text in texts:
            encoding = tokenizer(
                text,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            input_ids = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            pred = torch.argmax(logits, dim=1)
            
            predictions.append(pred.item())
    
    return predictions

def label_to_intent(label):
    """Convert label to intent description"""
    intent_map = {
        0: 'text_only (informational/reference)',
        1: 'dynamic_retrieval_intent (action required)',
    }
    return intent_map.get(label, 'unknown')
'''

utilities_file = os.path.join(model_path, 'utilities.py')
with open(utilities_file, 'w') as f:
    f.write(utilities_code)
print(f"✓ Utility functions saved: {utilities_file}")

# ============= SUMMARY EXPORT =============
print("\n" + "=" * 70)
print("EXPORT SUMMARY")
print("=" * 70)

# List all files in export directory
print(f"\nExported files and directories in: {model_path}")
for root, dirs, files in os.walk(model_path):
    level = root.replace(model_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # Convert to MB
        print(f"{subindent}{file} ({file_size:.2f} MB)")

print(f"\n✓ All models exported successfully!")
print(f"✓ Export directory: {model_path}")
print(f"\nTo use the exported models:")
print(f"  from utilities import load_baseline_model, load_finetuned_model")
print(f"  baseline_model, vectorizer = load_baseline_model('{model_path}')")
print(f"  finetuned_model, tokenizer = load_finetuned_model('{model_path}')")

### Loading and Using Exported Models

The exported models can be loaded and used in other scripts or notebooks using the provided utility functions.

In [ ]:
# ============= EXAMPLE: LOADING AND USING EXPORTED MODELS =============
print("\n" + "=" * 70)
print("EXAMPLE: LOADING AND USING EXPORTED MODELS")
print("=" * 70)

# Example 1: Load baseline model
print("\n1. Loading Baseline Model...")
try:
    baseline_loaded, vectorizer_loaded = joblib.load(baseline_model_file), joblib.load(baseline_vectorizer_file)
    print(f"   ✓ Baseline model loaded successfully")
    print(f"   Model type: {type(baseline_loaded).__name__}")
except Exception as e:
    print(f"   ✗ Error loading baseline model: {e}")

# Example 2: Load fine-tuned model
print("\n2. Loading Fine-tuned Model...")
try:
    finetuned_loaded = AutoModelForSequenceClassification.from_pretrained(
        os.path.join(finetuned_model_dir, 'model')
    )
    tokenizer_loaded = AutoTokenizer.from_pretrained(
        os.path.join(finetuned_model_dir, 'tokenizer')
    )
    print(f"   ✓ Fine-tuned model loaded successfully")
    print(f"   Model type: {type(finetuned_loaded).__name__}")
    print(f"   Model config: {finetuned_loaded.config}")
except Exception as e:
    print(f"   ✗ Error loading fine-tuned model: {e}")

# Example 3: Load metadata
print("\n3. Loading Metadata...")
try:
    with open(metadata_file, 'r') as f:
        metadata_loaded = json.load(f)
    print(f"   ✓ Metadata loaded successfully")
    print(f"   Model version: {metadata_loaded['model_version']}")
    print(f"   Dataset: {metadata_loaded['dataset']}")
    print(f"   Test set accuracy: {metadata_loaded['performance_metrics']['finetuned']['accuracy']:.4f}")
except Exception as e:
    print(f"   ✗ Error loading metadata: {e}")

# Example 4: Make predictions with both models
print("\n4. Making Predictions on Sample Queries...")
sample_queries = [
    "What is the meaning of life?",
    "Book me a flight to New York",
    "Tell me a joke",
    "What's the weather like today?",
]

print("\n   Sample Queries:")
for i, query in enumerate(sample_queries, 1):
    print(f"   {i}. {query}")

print("\n   Baseline Model Predictions:")
X_sample = tfidf_vectorizer.transform(sample_queries)
baseline_preds = baseline_model.predict(X_sample)
for i, (query, pred) in enumerate(zip(sample_queries, baseline_preds), 1):
    intent = 'text_only' if pred == 0 else 'dynamic_retrieval_intent'
    print(f"   {i}. [{intent}] {query}")

print("\n   Fine-tuned Model Predictions:")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
finetuned_loaded.to(device)
finetuned_loaded.eval()

finetuned_preds = []
with torch.no_grad():
    for query in sample_queries:
        encoding = tokenizer_loaded(
            query,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        outputs = finetuned_loaded(input_ids=input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1).item()
        finetuned_preds.append(pred)

for i, (query, pred) in enumerate(zip(sample_queries, finetuned_preds), 1):
    intent = 'text_only' if pred == 0 else 'dynamic_retrieval_intent'
    print(f"   {i}. [{intent}] {query}")

print("\n" + "=" * 70)
print("✓ Export and example usage completed successfully!")
print("=" * 70)